# V108 - Bagging under exact-text folds

Pre-registered robustness check: the bagging adoption rests on six-word-skeleton folds. Exact-text grouping is stricter (identical first messages share folds, singletons stand alone). Bagged Ridge faces single Ridge under this harsher split with a fresh seed. Same floor and worst-fold locks. A pass means the finding does not depend on fold definition.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 3334
N_SPLITS = 5
GATE_FLOOR = 0.002
BAGS = 10
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v108")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Same duel, harsher folds

Groups are exact first-chat texts. OOB bagging is unchanged. Only the verdict folds differ.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
firsts = [str(by_user.get(u, [""])[0]) if by_user.get(u, []) else "NO_CHAT" for u in list(train["user_id"])]
uniq, inv = np.unique(firsts, return_inverse=True)
print(f"exact-text groups: {len(uniq)} for {n_train} users")
fold_groups = np.array(firsts)
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

r = np.random.RandomState(SEED)
acc = np.zeros_like(Y)
cnt = np.zeros(n_train)
for b in range(BAGS):
    idx = r.choice(n_train, n_train, replace=True)
    oob = np.setdiff1d(np.arange(n_train), np.unique(idx))
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[idx], Y[idx, j])
        acc[oob, j] += m.predict(W[oob])
    cnt[oob] += 1
oof_b = acc / np.maximum(cnt[:, None], 1)
oof_s = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_s[va, j] = m.predict(W[va])
    fids[va] = fi + 1
base = ndcg5(Y, oof_s)
d = ndcg5(Y, oof_b) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fids == k], oof_b[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
print(f"single={base.mean():.5f} bagged={ndcg5(Y, oof_b).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ROBUST: adoption holds under exact-text folds" if (lo > GATE_FLOOR and wf >= 0) else "FRAGILE: adoption depends on fold definition")

exact-text groups: 1106 for 4000 users


single=0.63881 bagged=0.63722 delta=-0.00160 CI95=[-0.00459,+0.00133] worst-fold=-0.00538
FRAGILE: adoption depends on fold definition
